# 02 — Feature Analysis & Ablation

This notebook analyses the discriminative power of the extracted DSP features:
- Feature vector inspection
- Per-feature class separability (ANOVA F-score)
- Feature importance from a quick Random Forest fit
- UMAP/t-SNE visualisation of the feature space
- Ablation: which feature groups matter most?

In [ ]:
import os, sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score

from src.features import extract_features

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

In [ ]:
DATA_ROOT  = os.environ.get('DATA_ROOT', '../data')
TRAIN_DIR  = os.path.join(DATA_ROOT, 'train')
AUDIO_DIR  = os.path.join(TRAIN_DIR, 'audio')
LABELS_CSV = os.path.join(TRAIN_DIR, 'labels.csv')

df = pd.read_csv(LABELS_CSV)
df['clip_id'] = df['clip_id'].astype(str)

classes      = sorted(df['label'].unique())
label_to_idx = {c: i for i, c in enumerate(classes)}
print(f'{len(df)} clips | {len(classes)} classes')

## 1. Build Feature Matrix

Extract clean features for all training clips (cache to avoid re-extracting).

In [ ]:
CACHE = '../feature_cache.npz'

if os.path.exists(CACHE):
    data = np.load(CACHE)
    X, y = data['X'], data['y']
    print(f'Loaded from cache: X={X.shape}')
else:
    X, y = [], []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        path = os.path.join(AUDIO_DIR, f"{row['clip_id']}.wav")
        X.append(extract_features(path))
        y.append(label_to_idx[row['label']])
    X = np.stack(X)
    y = np.array(y, dtype=np.int64)
    np.savez(CACHE, X=X, y=y)
    print(f'Saved cache: X={X.shape}')

## 2. Feature Dimensionality & Basic Stats

In [ ]:
print(f'Feature vector length : {X.shape[1]}')
print(f'Mean feature value    : {X.mean():.4f}')
print(f'Std  feature value    : {X.std():.4f}')
print(f'Min / Max             : {X.min():.4f} / {X.max():.4f}')

# Distribution of a few features
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, idx in zip(axes, [0, 20, 100, 200]):
    ax.hist(X[:, idx], bins=40, color='steelblue', edgecolor='white')
    ax.set_title(f'Feature {idx}')
plt.suptitle('Distribution of Selected Features')
plt.tight_layout()
plt.show()

## 3. Class Separability — ANOVA F-score

Higher F-score = feature is more discriminative across classes.

In [ ]:
f_scores, _ = f_classif(X, y)

plt.figure(figsize=(14, 4))
plt.plot(f_scores, linewidth=0.8, color='darkorange')
plt.fill_between(range(len(f_scores)), f_scores, alpha=0.3, color='darkorange')
plt.xlabel('Feature Index')
plt.ylabel('ANOVA F-Score')
plt.title('Per-Feature Class Separability (ANOVA F-Score)')
plt.tight_layout()
plt.savefig('feature_separability.png', dpi=150)
plt.show()

top_k = np.argsort(f_scores)[::-1][:10]
print('Top 10 most discriminative feature indices:', top_k)

## 4. Feature Importance from Random Forest

In [ ]:
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_tr, y_tr)

importances = rf.feature_importances_

plt.figure(figsize=(14, 4))
plt.plot(importances, linewidth=0.8, color='steelblue')
plt.fill_between(range(len(importances)), importances, alpha=0.3, color='steelblue')
plt.xlabel('Feature Index')
plt.ylabel('Importance')
plt.title('Random Forest Feature Importances')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()

f1_val = f1_score(y_va, rf.predict(X_va), average='macro')
print(f'Quick RF validation Macro-F1: {f1_val:.4f}')

## 5. 2D Feature Space Visualisation (t-SNE)

Reduce to 2D to see whether feature clusters correspond to sound classes.

In [ ]:
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

# Scale first for better t-SNE convergence
X_scaled = StandardScaler().fit_transform(X)

tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
X_2d = tsne.fit_transform(X_scaled)

plt.figure(figsize=(12, 10))
scatter = plt.scatter(X_2d[:, 0], X_2d[:, 1], c=y, cmap='tab20', s=15, alpha=0.7)
plt.colorbar(scatter, label='Class Index')
plt.title('t-SNE Projection of Feature Space (50 Classes)')
plt.tight_layout()
plt.savefig('tsne_feature_space.png', dpi=150)
plt.show()

## 6. Feature Group Ablation

Train a quick RF using only one feature group at a time to see which contributes most.

In [ ]:
# NOTE: These slice boundaries depend on your feature vector layout.
# Adjust them if you change extract_features().
# Each feature group uses _segment_stats which gives:
#   global (2 * n_feat) + 4 segments * 2 * n_feat = 10 * n_feat values

feature_groups = {
    'MFCC'              : (0,   200),
    'MFCC Delta'        : (200, 400),
    'MFCC Delta2'       : (400, 600),
    'Spectral Centroid' : (600, 610),
    'Bandwidth'         : (610, 620),
    'Rolloff'           : (620, 630),
    'ZCR'               : (630, 640),
    'Log-Mel'           : (640, 1040),
    'Contrast'          : (1040, 1110),
    'RMS'               : (1110, 1120),
}

results = {}
for name, (start, end) in feature_groups.items():
    X_group = X[:, start:end]
    Xtr, Xva, ytr, yva = train_test_split(X_group, y, test_size=0.2, stratify=y, random_state=42)
    rf_g = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
    rf_g.fit(Xtr, ytr)
    score = f1_score(yva, rf_g.predict(Xva), average='macro')
    results[name] = score
    print(f'{name:25s}  F1 = {score:.4f}')

plt.figure(figsize=(10, 5))
bars = plt.barh(list(results.keys()), list(results.values()), color='steelblue')
plt.xlabel('Macro F1 (validation)')
plt.title('Feature Group Ablation')
plt.tight_layout()
plt.savefig('feature_ablation.png', dpi=150)
plt.show()